# Модуль 10.1 — Агент руками: цикл ReAct без фреймворка

Что прячет `CodeAgent.run()` из модуля 10? Соберём ту же петлю
**Thought → Action → Observation** в ~50 строк голого Python.

**Run all.** Нужен `HF_TOKEN` (Kaggle: Add-ons → Secrets; локально: переменная окружения).
Лекция: https://github.com/ITrubnikov/Train_of_Thought-homework/tree/main/notebooks/module-10-1-agent-by-hand

In [ ]:
%pip -q install huggingface_hub

In [ ]:
import os, json, re
from huggingface_hub import InferenceClient

# OpenAI-совместимый chat-интерфейс HF Inference. Тот же HF_TOKEN, что в модулях 10 и 10.7.
client = InferenceClient(model="Qwen/Qwen2.5-Coder-32B-Instruct", token=os.environ["HF_TOKEN"])

## 1. Дамми-тул

Учебный инструмент — обычная Python-функция с зашитым ответом. Реальная логика
тут не важна: важно, что **код** вызывает её, а не модель «придумывает» результат.

In [ ]:
def get_weather(location: str) -> str:
    return f"the weather in {location} is sunny with low temperatures.\n"

TOOLS = {"get_weather": get_weather}

## 2. Демо: без стоп-токена модель галлюцинирует

Попросим модель в формате Thought/Action/Observation — но **не** остановим её.
Она сама допишет `Observation:` с выдуманной погодой, не вызвав никакого тула.

In [ ]:
NAIVE_PROMPT = """Answer using the tool get_weather(location). Use this format:
Thought: ...
Action:
```
{"action": "get_weather", "action_input": {"location": "..."}}
```
Observation: the result of the action.
Thought: I now know the final answer.
Final Answer: ...
"""

out = client.chat.completions.create(
    messages=[{"role": "system", "content": NAIVE_PROMPT},
              {"role": "user", "content": "What's the weather in London?"}],
    max_tokens=300,
)
print(out.choices[0].message.content)  # модель сама придумает Observation

## 3. Фикс: `stop=["Observation:"]`

Стоп-токен обрывает генерацию ровно перед `Observation:` — управление
возвращается нашему коду, который вызывает **настоящий** тул и подаёт результат.

In [ ]:
SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:
get_weather: Get the current weather in a given location.

To call a tool, output a JSON blob with an "action" key (the tool name) and an
"action_input" key (the arguments). Use EXACTLY this format:

Question: the input question
Thought: think about the single next action. Only one action at a time.
Action:
```
{"action": "get_weather", "action_input": {"location": "London"}}
```
Observation: the result of the action.

(repeat Thought/Action/Observation as needed)

End with:
Thought: I now know the final answer.
Final Answer: the answer to the original question.
"""

## 4. Петля — это и есть «агент руками»

LLM → парсим JSON-действие → код вызывает тул → реальный Observation обратно в
контекст → снова LLM. Повторяем, пока не появится `Final Answer:`.

In [ ]:
def run_agent(question: str, max_steps: int = 5) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    for step in range(max_steps):
        out = client.chat.completions.create(
            messages=messages, max_tokens=300, stop=["Observation:"]
        )
        text = out.choices[0].message.content
        print(f"--- step {step} ---\n{text}")
        if "Final Answer:" in text:
            return text.split("Final Answer:")[-1].strip()
        action = json.loads(re.search(r"\{.*\}", text, re.DOTALL).group())
        result = TOOLS[action["action"]](**action["action_input"])
        messages.append({"role": "assistant", "content": text + "Observation: " + result})
    return "(достигнут max_steps без Final Answer)"

print(run_agent("What's the weather in London?"))

## Задачи

1. **Второй тул.** Добавь `get_time(timezone)` в `TOOLS` и в `SYSTEM_PROMPT`,
   задай вопрос, требующий двух шагов (погода + время). Покажи трейс.
2. **Сломай формат.** Убери из промпта требование JSON/формата — посмотри, как
   падает `json.loads`. В md-ячейке опиши, почему формат — это контракт.
3. *(Опц.)* **Смени провайдера** на MiniMax (OpenAI-совместимый
   `api.minimax.io/v1`) — поменяй только строку создания клиента.

## Что сдать

Ссылка на твой ноутбук с трейсом прогона на **два шага** (видно Thought/Action/
Observation дважды и финальный ответ). Формат: `[Модуль 10.1, ДЗ 1] {ссылка}`.